In [ ]:
!pip install -q yfinance scikit-learn xgboost lightgbm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

NEWS_FILE = Path("/content/drive/MyDrive/bigquery/bq-results-2.csv")
OUT_DIR = Path("/content/drive/MyDrive/oil_news_project_outputs")
OUT_DIR.mkdir(exist_ok=True)

EVENT_START = "2026-02-15"
EVENT_END = "2026-04-27"   # 结束日用开区间，多加一天

print("News file exists:", NEWS_FILE.exists())
print("Output folder:", OUT_DIR)

News file exists: True
Output folder: /content/drive/MyDrive/oil_news_project_outputs


In [ ]:
preview = pd.read_csv(NEWS_FILE, nrows=5, low_memory=False)
print(preview.columns.tolist())
display(preview.head())

['published_utc', 'date_utc', 'source', 'url', 'title', 'query_bucket', 'relevance_score', 'V2Themes', 'V2Tone']


,published_utc,date_utc,source,url,title,query_bucket,relevance_score,V2Themes,V2Tone
0,2026-03-11 09:30:00 UTC,2026-03-11,winnipegfreepress.com,https://www.winnipegfreepress.com/arts-and-lif...,Analysis: Iran war becomes a contest of who ca...,P1_military_escalation,2,"ARMEDCONFLICT,69;ARMEDCONFLICT,642;ARMEDCONFLI...","-3.41880341880342,2.48640248640249,5.905205905..."
1,2026-03-11 09:30:00 UTC,2026-03-11,indiatimes.com,https://timesofindia.indiatimes.com/life-style...,"UAE flights update: Emirates, Etihad run reduc...",P2_energy_supply_chain,2,"TAX_FNCACT_OFFICIAL,1228;TAX_FNCACT_OFFICIAL,4...","0.15015015015015,1.8018018018018,1.65165165165..."
2,2026-03-11 09:30:00 UTC,2026-03-11,walesonline.co.uk,https://www.walesonline.co.uk/news/world-news/...,Thousands of people get energy bills email aft...,P1_iran_oil_direct,2,"ECON_OILPRICE,2426;TAX_FNCACT_CHANCELLOR,797;T...","-4.77064220183486,1.10091743119266,5.871559633..."
3,2026-03-11 09:30:00 UTC,2026-03-11,constructionenquirer.com,https://www.constructionenquirer.com/2026/03/1...,Forterra 'insulated' from Iran war gas price s...,P1_military_escalation,2,"TAX_MILITARY_TITLE_OFFICER,942;TAX_FNCACT_OFFI...","0.442477876106194,3.98230088495575,3.539823008..."
4,2026-03-11 09:30:00 UTC,2026-03-11,perthnow.com.au,https://www.perthnow.com.au/lifestyle/motoring...,2026 Volkswagen Amarok ditches four-cylinder d...,P2_energy_supply_chain,2,NaN,"0.936329588014981,1.68539325842697,0.749063670..."


In [ ]:
available_cols = preview.columns.tolist()

wanted_cols = [
    "published_utc",
    "date_utc",
    "source",
    "url",
    "title",
    "query_bucket",
    "relevance_score",
    "V2Themes",
    "V2Tone"
]

usecols = [c for c in wanted_cols if c in available_cols]

news = pd.read_csv(
    NEWS_FILE,
    usecols=usecols,
    low_memory=False
)

print("Raw news shape:", news.shape)
print(news.columns.tolist())
display(news.head())

Raw news shape: (210848, 9)
['published_utc', 'date_utc', 'source', 'url', 'title', 'query_bucket', 'relevance_score', 'V2Themes', 'V2Tone']


,published_utc,date_utc,source,url,title,query_bucket,relevance_score,V2Themes,V2Tone
0,2026-03-11 09:30:00 UTC,2026-03-11,winnipegfreepress.com,https://www.winnipegfreepress.com/arts-and-lif...,Analysis: Iran war becomes a contest of who ca...,P1_military_escalation,2,"ARMEDCONFLICT,69;ARMEDCONFLICT,642;ARMEDCONFLI...","-3.41880341880342,2.48640248640249,5.905205905..."
1,2026-03-11 09:30:00 UTC,2026-03-11,indiatimes.com,https://timesofindia.indiatimes.com/life-style...,"UAE flights update: Emirates, Etihad run reduc...",P2_energy_supply_chain,2,"TAX_FNCACT_OFFICIAL,1228;TAX_FNCACT_OFFICIAL,4...","0.15015015015015,1.8018018018018,1.65165165165..."
2,2026-03-11 09:30:00 UTC,2026-03-11,walesonline.co.uk,https://www.walesonline.co.uk/news/world-news/...,Thousands of people get energy bills email aft...,P1_iran_oil_direct,2,"ECON_OILPRICE,2426;TAX_FNCACT_CHANCELLOR,797;T...","-4.77064220183486,1.10091743119266,5.871559633..."
3,2026-03-11 09:30:00 UTC,2026-03-11,constructionenquirer.com,https://www.constructionenquirer.com/2026/03/1...,Forterra 'insulated' from Iran war gas price s...,P1_military_escalation,2,"TAX_MILITARY_TITLE_OFFICER,942;TAX_FNCACT_OFFI...","0.442477876106194,3.98230088495575,3.539823008..."
4,2026-03-11 09:30:00 UTC,2026-03-11,perthnow.com.au,https://www.perthnow.com.au/lifestyle/motoring...,2026 Volkswagen Amarok ditches four-cylinder d...,P2_energy_supply_chain,2,NaN,"0.936329588014981,1.68539325842697,0.749063670..."


In [ ]:
news.columns = [c.strip() for c in news.columns]

if "published_utc" not in news.columns:
    raise ValueError("没有 published_utc 列，请检查 BigQuery 导出的列名。")

if "title" not in news.columns:
    raise ValueError("没有 title 列，请检查 BigQuery 导出的列名。")

news["published_utc"] = pd.to_datetime(
    news["published_utc"],
    errors="coerce",
    utc=True
)

news = news.dropna(subset=["published_utc", "title"]).copy()

# 限制事件窗口
news = news[
    (news["published_utc"] >= pd.Timestamp(EVENT_START, tz="UTC")) &
    (news["published_utc"] < pd.Timestamp(EVENT_END, tz="UTC"))
].copy()

# 补缺失列
for col in ["source", "url", "query_bucket", "relevance_score"]:
    if col not in news.columns:
        news[col] = np.nan

news["relevance_score"] = pd.to_numeric(
    news["relevance_score"],
    errors="coerce"
).fillna(1)

def clean_title_text(x):
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_for_dedup(x):
    x = clean_title_text(x).lower()
    x = re.sub(r"http\S+", "", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

news["title_clean"] = news["title"].apply(clean_title_text)
news["title_norm"] = news["title_clean"].apply(normalize_for_dedup)

# 去掉过短标题
news = news[news["title_clean"].str.len() >= 20].copy()

# exact duplicate 去重
news = news.sort_values("published_utc")
news = news.drop_duplicates(subset=["title_norm"], keep="first").copy()

# 删除明显无关标题
irrelevant_patterns = [
    r"\bolive oil\b",
    r"\boil painting\b",
    r"\bcooking oil\b",
    r"\bessential oil\b",
    r"\bhair oil\b",
    r"\bmassage oil\b",
]

irrelevant_regex = re.compile("|".join(irrelevant_patterns), re.IGNORECASE)
news = news[~news["title_clean"].str.contains(irrelevant_regex, na=False)].copy()

news = news.reset_index(drop=True)

print("Cleaned news shape:", news.shape)
display(news[["published_utc", "source", "title_clean", "query_bucket", "relevance_score"]].head())

Cleaned news shape: (209923, 11)


,published_utc,source,title_clean,query_bucket,relevance_score
0,2026-02-15 00:00:00+00:00,masdark.com,2026 Winter Olympics: Mikhail Shaidorov wins m...,P3_macro_financial,1
1,2026-02-15 00:00:00+00:00,nwaonline.com,Olympic gold medalist Noah Lyles runs personal...,P3_macro_financial,1
2,2026-02-15 00:00:00+00:00,640wxsm.com,Charley Hull uses late charge to win PIF Saudi...,P2_energy_supply_chain,2
3,2026-02-15 00:00:00+00:00,valleynewslive.com,Jordan Stolz wins his 2nd Olympic speedskating...,P3_macro_financial,1
4,2026-02-15 00:00:00+00:00,fortworthbusiness.com,Fort Worth Chamber Summit to Address Workforce...,P2_energy_supply_chain,2


In [ ]:
topic_keywords = {
    "hormuz_shipping": [
        "hormuz", "strait of hormuz", "persian gulf",
        "tanker", "vessel", "shipping", "shipping lane",
        "blockade", "maritime", "gulf waters"
    ],

    "military_escalation": [
        "attack", "attacks", "strike", "strikes", "missile", "missiles",
        "drone", "drones", "troops", "military", "war", "conflict",
        "retaliation", "retaliate", "escalation", "airstrike",
        "bombing", "shelling"
    ],

    "sanctions_exports": [
        "sanction", "sanctions", "embargo", "export", "exports",
        "iranian oil", "oil exports", "crude exports",
        "oil revenue", "production", "output", "supply cut"
    ],

    "diplomacy_ceasefire": [
        "talks", "negotiation", "negotiations", "ceasefire",
        "peace", "deal", "agreement", "diplomacy", "diplomatic",
        "truce", "resume talks", "peace talks"
    ],

    "opec_gulf_supply": [
        "opec", "opec+", "saudi", "saudi arabia", "uae",
        "iraq", "kuwait", "qatar", "gulf producers",
        "production increase", "production cut", "spare capacity"
    ],

    "inventory_eia": [
        "inventory", "inventories", "stockpile", "stockpiles",
        "eia", "api", "refinery", "refineries", "gasoline",
        "diesel", "crude stocks", "drawdown", "build"
    ],

    "macro_dollar_fed": [
        "dollar", "fed", "federal reserve", "inflation",
        "interest rate", "rates", "rate cut", "rate hike",
        "yields", "treasury", "stock market", "risk appetite"
    ],

    "demand_china_growth": [
        "demand", "china", "chinese demand", "global growth",
        "recession", "slowdown", "economic growth",
        "manufacturing", "factory activity", "asia demand"
    ],
}

topic_priority = [
    "hormuz_shipping",
    "military_escalation",
    "sanctions_exports",
    "diplomacy_ceasefire",
    "inventory_eia",
    "opec_gulf_supply",
    "macro_dollar_fed",
    "demand_china_growth",
]

def assign_topic_rule(title):
    text = str(title).lower()

    for topic in topic_priority:
        for kw in topic_keywords[topic]:
            pattern = r"\b" + re.escape(kw.lower()) + r"\b"
            if re.search(pattern, text):
                return topic

    return "other_oil_relevant"

news["topic"] = news["title_clean"].apply(assign_topic_rule)

print(news["topic"].value_counts())

news.to_csv(OUT_DIR / "news_clean_topic_rule_based.csv", index=False)

topic
military_escalation    100012
hormuz_shipping         36761
other_oil_relevant      23241
macro_dollar_fed        19059
opec_gulf_supply        15310
inventory_eia            7490
sanctions_exports        3616
diplomacy_ceasefire      3261
demand_china_growth      1173
Name: count, dtype: int64


In [ ]:
def aggregate_text_features(df, freq="1h"):
    temp = df.copy()
    temp["bucket"] = temp["published_utc"].dt.floor(freq)

    temp["relevance_score"] = pd.to_numeric(
        temp["relevance_score"],
        errors="coerce"
    ).fillna(1)

    # 所有新闻聚合
    all_agg = temp.groupby("bucket").agg(
        all_news_count=("title_clean", "size"),
        all_source_count=("source", "nunique"),
        all_relevance_sum=("relevance_score", "sum"),
        all_relevance_mean=("relevance_score", "mean")
    )

    # 分 topic 聚合
    topic_agg = temp.groupby(["bucket", "topic"]).agg(
        count=("title_clean", "size"),
        source_count=("source", "nunique"),
        relevance_sum=("relevance_score", "sum"),
        relevance_mean=("relevance_score", "mean")
    )

    topic_wide = topic_agg.unstack("topic")
    topic_wide.columns = [
        f"{topic}_{metric}"
        for metric, topic in topic_wide.columns
    ]

    features = all_agg.join(topic_wide, how="outer").fillna(0)

    # log1p 版本，降低极端新闻数量影响
    for c in list(features.columns):
        if c.endswith("_count") or c.endswith("_sum") or c in ["all_news_count", "all_relevance_sum"]:
            features[f"log1p_{c}"] = np.log1p(features[c])

    features.index.name = "timestamp_utc"
    features = features.reset_index()

    return features

text_1h = aggregate_text_features(news, freq="1h")
text_4h = aggregate_text_features(news, freq="4h")

print("Text 1h:", text_1h.shape)
print("Text 4h:", text_4h.shape)

display(text_4h.head())

text_1h.to_csv(OUT_DIR / "text_features_1h_topic_count.csv", index=False)
text_4h.to_csv(OUT_DIR / "text_features_4h_topic_count.csv", index=False)

Text 1h: (1702, 71)
Text 4h: (426, 71)


,timestamp_utc,all_news_count,all_source_count,all_relevance_sum,all_relevance_mean,demand_china_growth_count,diplomacy_ceasefire_count,hormuz_shipping_count,inventory_eia_count,macro_dollar_fed_count,...,log1p_sanctions_exports_source_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 00:00:00+00:00,126,87,186,1.476190,5.0,1.0,2.0,1.0,13.0,...,2.079442,2.484907,1.386294,1.945910,1.098612,2.639057,3.806662,2.944439,4.262680,2.995732
1,2026-02-15 04:00:00+00:00,119,84,191,1.605042,2.0,0.0,3.0,5.0,27.0,...,1.791759,1.945910,0.000000,2.302585,2.397895,3.332205,3.871201,3.433987,3.806662,2.944439
2,2026-02-15 08:00:00+00:00,128,86,234,1.828125,5.0,2.0,3.0,3.0,14.0,...,2.302585,2.772589,1.609438,2.302585,1.945910,2.708050,4.007333,4.356709,3.401197,3.295837
3,2026-02-15 12:00:00+00:00,182,127,317,1.741758,2.0,4.0,16.0,5.0,25.0,...,2.079442,1.609438,2.079442,4.158883,2.397895,3.258097,3.637586,4.488636,4.158883,3.044522
4,2026-02-15 16:00:00+00:00,176,125,306,1.738636,0.0,2.0,22.0,2.0,22.0,...,1.098612,0.000000,1.609438,4.394449,1.609438,3.135494,4.060443,3.970292,4.406719,1.945910


In [ ]:
import yfinance as yf

tickers = {
    "WTI": "CL=F",
    "BRENT": "BZ=F",
    "GOLD": "GC=F",
    "NATGAS": "NG=F",
    "SPY": "SPY",
    "UUP": "UUP",
    "VIX": "^VIX"
}

raw = yf.download(
    list(tickers.values()),
    start=EVENT_START,
    end=EVENT_END,
    interval="1h",
    auto_adjust=False,
    progress=True,
    group_by="ticker",
    threads=True
)

display(raw.head())
print(raw.shape)

[*********************100%***********************]  7 of 7 completed


Ticker                     SPY                                 ^VIX           \
Price                     Open High Low Close Adj Close Volume Open High Low   
Datetime                                                                       
2026-02-15 23:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 00:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 01:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 02:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 03:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   

Ticker                           ...         GC=F                            \
Price                     Close  ...          Low        Close    Adj Close   
Datetime                         ...                                          
2026-02-15 23:00:00+00:00   NaN  ...  5031.600098  5038.500000  5038.500000   
2026-02-16 00:00:00+00:00   NaN  ...  5018.200195  5035.000000  5035.000000   
2026-02-16 01:00:00+00:00   NaN  ...  5026.399902  5047.600098  5047.600098   
2026-02-16 02:00:00+00:00   NaN  ...  5011.299805  5014.799805  5014.799805   
2026-02-16 03:00:00+00:00   NaN  ...  4986.000000  5007.200195  5007.200195   

Ticker                                  BZ=F                                   \
Price                      Volume       Open       High        Low      Close   
Datetime                                                                        
2026-02-15 23:00:00+00:00   917.0  67.750000  67.760002  67.699997  67.739998   
2026-02-16 00:00:00+00:00  1713.0  67.750000  67.750000  67.709999  67.730003   
2026-02-16 01:00:00+00:00  1387.0  67.730003  67.779999  67.690002  67.690002   
2026-02-16 02:00:00+00:00  2108.0  67.639999  67.800003  67.629997  67.739998   
2026-02-16 03:00:00+00:00  2652.0  67.790001  67.820000  67.690002  67.820000   

Ticker                                       
Price                      Adj Close Volume  
Datetime                                     
2026-02-15 23:00:00+00:00  67.739998   64.0  
2026-02-16 00:00:00+00:00  67.730003   92.0  
2026-02-16 01:00:00+00:00  67.690002   81.0  
2026-02-16 02:00:00+00:00  67.739998  179.0  
2026-02-16 03:00:00+00:00  67.820000  102.0  

[5 rows x 42 columns]

(1447, 42)


In [ ]:
def extract_close(data, ticker_map):
    close = pd.DataFrame()

    for name, ticker in ticker_map.items():
        try:
            if isinstance(data.columns, pd.MultiIndex):
                if (ticker, "Close") in data.columns:
                    close[name] = data[(ticker, "Close")]
                elif ("Close", ticker) in data.columns:
                    close[name] = data[("Close", ticker)]
            else:
                close[name] = data["Close"]
        except Exception as e:
            print("Cannot extract:", name, ticker, e)

    close.index = pd.to_datetime(close.index)

    if close.index.tz is None:
        close.index = close.index.tz_localize("UTC")
    else:
        close.index = close.index.tz_convert("UTC")

    close = close.dropna(how="all")

    return close

px_1h = extract_close(raw, tickers)

print(px_1h.shape)
display(px_1h.head())

px_1h.to_csv(OUT_DIR / "yahoo_market_prices_1h.csv")

(1447, 7)


,WTI,BRENT,GOLD,NATGAS,SPY,UUP,VIX
Datetime,,,,,,,
2026-02-15 23:00:00+00:00,62.900002,67.739998,5038.500000,3.038,NaN,NaN,NaN
2026-02-16 00:00:00+00:00,62.860001,67.730003,5035.000000,3.016,NaN,NaN,NaN
2026-02-16 01:00:00+00:00,62.869999,67.690002,5047.600098,3.015,NaN,NaN,NaN
2026-02-16 02:00:00+00:00,62.900002,67.739998,5014.799805,3.011,NaN,NaN,NaN
2026-02-16 03:00:00+00:00,62.910000,67.820000,5007.200195,3.022,NaN,NaN,NaN


In [ ]:
def make_price_features(px, freq="1h"):
    p = px.copy()

    # 转成固定频率
    p = p.resample(freq, label="left", closed="left").last()
    p = p.dropna(subset=["WTI"]).ffill()

    feat = pd.DataFrame(index=p.index)
    feat["wti_close"] = p["WTI"]

    # WTI 当前收益率
    feat["wti_ret_0"] = np.log(p["WTI"]).diff()

    # WTI lag returns
    for lag in [1, 2, 3, 6, 12]:
        feat[f"wti_ret_lag{lag}"] = feat["wti_ret_0"].shift(lag)

    # WTI rolling volatility
    for window in [3, 6, 12]:
        feat[f"wti_vol_{window}"] = feat["wti_ret_0"].rolling(window).std()

    # 其他市场变量收益率
    for col in p.columns:
        if col == "WTI":
            continue

        ret = np.log(p[col]).diff()
        name = col.lower()

        feat[f"{name}_ret_0"] = ret

        for lag in [1, 2, 3]:
            feat[f"{name}_ret_lag{lag}"] = ret.shift(lag)

    # 时间控制变量
    feat["hour"] = feat.index.hour
    feat["dayofweek"] = feat.index.dayofweek
    feat["hour_sin"] = np.sin(2 * np.pi * feat["hour"] / 24)
    feat["hour_cos"] = np.cos(2 * np.pi * feat["hour"] / 24)

    # 下一时段收益率和涨跌方向
    feat["future_ret_next"] = np.log(p["WTI"].shift(-1)) - np.log(p["WTI"])
    feat["target_direction"] = (feat["future_ret_next"] > 0).astype(int)

    feat.index.name = "timestamp_utc"
    feat = feat.reset_index()

    return feat

price_1h = make_price_features(px_1h, freq="1h")
price_4h = make_price_features(px_1h, freq="4h")

print("Price 1h:", price_1h.shape)
print("Price 4h:", price_4h.shape)

display(price_4h.head())

Price 1h: (1095, 41)
Price 4h: (297, 41)


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,vix_ret_0,vix_ret_lag1,vix_ret_lag2,vix_ret_lag3,hour,dayofweek,hour_sin,hour_cos,future_ret_next,target_direction
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,20,6,-0.866025,0.5,0.000159,1
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0.000000,1.0,0.000000,0
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4,0,0.866025,0.5,0.004599,1
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,NaN,NaN,NaN,NaN,4,1,0.866025,0.5,0.007409,1
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,NaN,NaN,NaN,NaN,8,1,0.866025,-0.5,-0.027870,0


In [ ]:
def merge_text_price(price_df, text_df):
    p = price_df.copy()
    t = text_df.copy()

    p["timestamp_utc"] = pd.to_datetime(p["timestamp_utc"], utc=True)
    t["timestamp_utc"] = pd.to_datetime(t["timestamp_utc"], utc=True)

    df = p.merge(t, on="timestamp_utc", how="left")

    text_cols = [c for c in t.columns if c != "timestamp_utc"]
    df[text_cols] = df[text_cols].fillna(0)

    df = df.replace([np.inf, -np.inf], np.nan)

    # 删除 target 缺失、价格 lag 缺失
    df = df.dropna(subset=["future_ret_next", "target_direction"]).copy()

    return df

df_1h = merge_text_price(price_1h, text_1h)
df_4h = merge_text_price(price_4h, text_4h)

print("Final 1h dataset:", df_1h.shape)
print("1h target distribution:")
print(df_1h["target_direction"].value_counts(normalize=True))

print("\nFinal 4h dataset:", df_4h.shape)
print("4h target distribution:")
print(df_4h["target_direction"].value_counts(normalize=True))

df_1h.to_csv(OUT_DIR / "direction_dataset_1h.csv", index=False)
df_4h.to_csv(OUT_DIR / "direction_dataset_4h.csv", index=False)

Final 1h dataset: (1094, 111)
1h target distribution:
target_direction
1    0.551188
0    0.448812
Name: proportion, dtype: float64

Final 4h dataset: (296, 111)
4h target distribution:
target_direction
1    0.574324
0    0.425676
Name: proportion, dtype: float64


In [ ]:
def infer_feature_groups(df):
    exclude_cols = [
        "timestamp_utc",
        "wti_close",
        "future_ret_next",
        "target_direction"
    ]

    numeric_cols = [
        c for c in df.columns
        if c not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    text_prefixes = [
        "all_",
        "log1p_all_",
        "hormuz_shipping_",
        "log1p_hormuz_shipping_",
        "military_escalation_",
        "log1p_military_escalation_",
        "sanctions_exports_",
        "log1p_sanctions_exports_",
        "diplomacy_ceasefire_",
        "log1p_diplomacy_ceasefire_",
        "opec_gulf_supply_",
        "log1p_opec_gulf_supply_",
        "inventory_eia_",
        "log1p_inventory_eia_",
        "macro_dollar_fed_",
        "log1p_macro_dollar_fed_",
        "demand_china_growth_",
        "log1p_demand_china_growth_",
        "other_oil_relevant_",
        "log1p_other_oil_relevant_",
    ]

    text_cols = [
        c for c in numeric_cols
        if any(c.startswith(prefix) for prefix in text_prefixes)
    ]

    financial_cols = [
        c for c in numeric_cols
        if c not in text_cols
    ]

    combined_cols = financial_cols + text_cols

    return text_cols, financial_cols, combined_cols

text_cols_1h, fin_cols_1h, combo_cols_1h = infer_feature_groups(df_1h)
text_cols_4h, fin_cols_4h, combo_cols_4h = infer_feature_groups(df_4h)

print("1h text:", len(text_cols_1h), "financial:", len(fin_cols_1h), "combined:", len(combo_cols_1h))
print("4h text:", len(text_cols_4h), "financial:", len(fin_cols_4h), "combined:", len(combo_cols_4h))

1h text: 70 financial: 37 combined: 107
4h text: 70 financial: 37 combined: 107


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except:
    HAS_XGB = False


def get_classifier(model_name):
    if model_name == "Baseline":
        return DummyClassifier(strategy="most_frequent")

    if model_name == "Logit_L2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                class_weight="balanced"
            ))
        ])

    if model_name == "Logit_L1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                max_iter=3000,
                class_weight="balanced"
            ))
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=3,
                min_samples_leaf=5,
                random_state=42,
                class_weight="balanced"
            ))
        ])

    if model_name == "XGBoost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=200,
                max_depth=2,
                learning_rate=0.03,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            ))
        ])

    raise ValueError("Unknown model name")


def evaluate_direction_accuracy(df, feature_cols, model_name, n_splits=5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    y = data["target_direction"].astype(int).values

    if len(feature_cols) == 0:
        X = pd.DataFrame({"constant": np.zeros(len(data))})
    else:
        X = data[feature_cols].replace([np.inf, -np.inf], np.nan)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = get_classifier(model_name)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X_test)[:, 1]
        else:
            prob = pred

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        y_prob_all.extend(prob)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    out = {
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

    try:
        out["auc"] = roc_auc_score(y_true_all, y_prob_all)
    except:
        out["auc"] = np.nan

    return out

In [ ]:
def run_direction_experiment(df, horizon_name):
    text_cols, fin_cols, combo_cols = infer_feature_groups(df)

    feature_sets = {
        "Baseline": [],
        "Text-only": text_cols,
        "Financial-only": fin_cols,
        "Text + Financial": combo_cols,
    }

    models = ["Baseline", "Logit_L2", "Logit_L1", "RandomForest"]

    if HAS_XGB:
        models.append("XGBoost")

    rows = []

    print("=" * 100)
    print("Horizon:", horizon_name)
    print("Dataset:", df.shape)
    print("Target distribution:")
    print(df["target_direction"].value_counts(normalize=True))
    print("Text features:", len(text_cols))
    print("Financial features:", len(fin_cols))

    for feature_set_name, cols in feature_sets.items():
        for model_name in models:

            if model_name == "Baseline" and feature_set_name != "Baseline":
                continue

            if model_name != "Baseline" and feature_set_name == "Baseline":
                continue

            print(f"Running {horizon_name} | {feature_set_name} | {model_name} | n_features={len(cols)}")

            res = evaluate_direction_accuracy(
                df,
                feature_cols=cols,
                model_name=model_name,
                n_splits=5
            )

            res["horizon"] = horizon_name
            res["feature_set"] = feature_set_name
            res["model"] = model_name
            res["n_features"] = len(cols)

            rows.append(res)

    results = pd.DataFrame(rows)

    results = results[
        [
            "horizon",
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "n_eval",
            "confusion_matrix"
        ]
    ].sort_values(["accuracy", "balanced_accuracy"], ascending=False)

    return results


results_1h = run_direction_experiment(df_1h, "1h_next_period")
results_4h = run_direction_experiment(df_4h, "4h_next_period")

Horizon: 1h_next_period
Dataset: (1094, 111)
Target distribution:
target_direction
1    0.551188
0    0.448812
Name: proportion, dtype: float64
Text features: 70
Financial features: 37
Running 1h_next_period | Baseline | Baseline | n_features=0
Running 1h_next_period | Text-only | Logit_L2 | n_features=70
Running 1h_next_period | Text-only | Logit_L1 | n_features=70
Running 1h_next_period | Text-only | RandomForest | n_features=70
Running 1h_next_period | Text-only | XGBoost | n_features=70
Running 1h_next_period | Financial-only | Logit_L2 | n_features=37
Running 1h_next_period | Financial-only | Logit_L1 | n_features=37
Running 1h_next_period | Financial-only | RandomForest | n_features=37
Running 1h_next_period | Financial-only | XGBoost | n_features=37
Running 1h_next_period | Text + Financial | Logit_L2 | n_features=107
Running 1h_next_period | Text + Financial | Logit_L1 | n_features=107
Running 1h_next_period | Text + Financial | RandomForest | n_features=107
Running 1h_next_per

In [ ]:
print("1h 下一小时涨跌预测结果")
display(
    results_1h[
        [
            "horizon",
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ]
)

1h 下一小时涨跌预测结果


,horizon,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred
0,1h_next_period,Baseline,Baseline,0,0.556044,0.500000,0.714689,0.500000,0.556044,1.000000
4,1h_next_period,Text-only,XGBoost,70,0.549451,0.513531,0.673046,0.506266,0.556044,0.821978
7,1h_next_period,Financial-only,RandomForest,37,0.539560,0.512870,0.644614,0.510967,0.556044,0.739560
8,1h_next_period,Financial-only,XGBoost,37,0.535165,0.501683,0.656934,0.504706,0.556044,0.798901
11,1h_next_period,Text + Financial,RandomForest,107,0.534066,0.510923,0.631304,0.496395,0.556044,0.707692
12,1h_next_period,Text + Financial,XGBoost,107,0.525275,0.491043,0.651050,0.494326,0.556044,0.804396
5,1h_next_period,Financial-only,Logit_L2,37,0.517582,0.516060,0.549744,0.501477,0.556044,0.515385
6,1h_next_period,Financial-only,Logit_L1,37,0.513187,0.512606,0.541882,0.506149,0.556044,0.506593
3,1h_next_period,Text-only,RandomForest,70,0.507692,0.490451,0.592727,0.485207,0.556044,0.652747
9,1h_next_period,Text + Financial,Logit_L2,107,0.498901,0.501756,0.513859,0.496331,0.556044,0.474725


In [ ]:
print("4h 下一 4 小时涨跌预测结果")
display(
    results_4h[
        [
            "horizon",
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ]
)

4h 下一 4 小时涨跌预测结果


,horizon,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred
1,4h_next_period,Text-only,Logit_L2,70,0.567347,0.532069,0.664557,0.522759,0.591837,0.697959
2,4h_next_period,Text-only,Logit_L1,70,0.563265,0.555000,0.619217,0.553931,0.591837,0.555102
0,4h_next_period,Baseline,Baseline,0,0.555102,0.500000,0.680352,0.500000,0.591837,0.800000
9,4h_next_period,Text + Financial,Logit_L2,107,0.546939,0.507069,0.654206,0.508655,0.591837,0.718367
10,4h_next_period,Text + Financial,Logit_L1,107,0.542857,0.500517,0.654321,0.513379,0.591837,0.730612
11,4h_next_period,Text + Financial,RandomForest,107,0.542857,0.477241,0.683616,0.446000,0.591837,0.853061
3,4h_next_period,Text-only,RandomForest,70,0.522449,0.464655,0.658892,0.468690,0.591837,0.808163
4,4h_next_period,Text-only,XGBoost,70,0.514286,0.462414,0.644776,0.434207,0.591837,0.775510
5,4h_next_period,Financial-only,Logit_L2,37,0.493878,0.466897,0.589404,0.465207,0.591837,0.640816
6,4h_next_period,Financial-only,Logit_L1,37,0.469388,0.444655,0.563758,0.455310,0.591837,0.624490


In [34]:
#安装 FinBERT 依赖
!pip install -q transformers torch accelerate

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

OUT_DIR = Path("/content/drive/MyDrive/oil_news_project_outputs")
NEWS_TOPIC_FILE = OUT_DIR / "news_clean_topic_rule_based.csv"

if "news" in globals():
    news_topic = news.copy()
    print("Using existing news variable.")
else:
    news_topic = pd.read_csv(NEWS_TOPIC_FILE)
    print("Loaded:", NEWS_TOPIC_FILE)

news_topic["published_utc"] = pd.to_datetime(news_topic["published_utc"], utc=True)

required_cols = ["published_utc", "title_clean", "topic", "relevance_score"]
for col in required_cols:
    if col not in news_topic.columns:
        raise ValueError(f"Missing column: {col}")

news_topic["relevance_score"] = pd.to_numeric(
    news_topic["relevance_score"], errors="coerce"
).fillna(1)

print(news_topic.shape)
print(news_topic["topic"].value_counts())
display(news_topic[["published_utc", "title_clean", "topic", "relevance_score"]].head())

Using existing news variable.
(209923, 12)
topic
military_escalation    100012
hormuz_shipping         36761
other_oil_relevant      23241
macro_dollar_fed        19059
opec_gulf_supply        15310
inventory_eia            7490
sanctions_exports        3616
diplomacy_ceasefire      3261
demand_china_growth      1173
Name: count, dtype: int64


,published_utc,title_clean,topic,relevance_score
0,2026-02-15 00:00:00+00:00,2026 Winter Olympics: Mikhail Shaidorov wins m...,other_oil_relevant,1
1,2026-02-15 00:00:00+00:00,Olympic gold medalist Noah Lyles runs personal...,other_oil_relevant,1
2,2026-02-15 00:00:00+00:00,Charley Hull uses late charge to win PIF Saudi...,opec_gulf_supply,2
3,2026-02-15 00:00:00+00:00,Jordan Stolz wins his 2nd Olympic speedskating...,other_oil_relevant,1
4,2026-02-15 00:00:00+00:00,Fort Worth Chamber Summit to Address Workforce...,other_oil_relevant,2


In [36]:
USE_FILTERED_NEWS = True

important_topics = [
    "diplomacy_ceasefire",
    "inventory_eia",
    "military_escalation",
    "hormuz_shipping",
    "sanctions_exports",
    "opec_gulf_supply",
    "macro_dollar_fed",
    "demand_china_growth",
    "other_oil_relevant"
]

if USE_FILTERED_NEWS:
    news_finbert_base = news_topic[
        (news_topic["topic"].isin(important_topics)) &
        (news_topic["relevance_score"] >= 1)
    ].copy()
else:
    news_finbert_base = news_topic.copy()

# 避免重复标题重复跑 FinBERT
news_finbert_base = news_finbert_base.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

print("News for FinBERT:", news_finbert_base.shape)
print(news_finbert_base["topic"].value_counts())

News for FinBERT: (209923, 12)
topic
military_escalation    100012
hormuz_shipping         36761
other_oil_relevant      23241
macro_dollar_fed        19059
opec_gulf_supply        15310
inventory_eia            7490
sanctions_exports        3616
diplomacy_ceasefire      3261
demand_china_growth      1173
Name: count, dtype: int64


In [37]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm

FINBERT_OUTPUT_FILE = OUT_DIR / "news_topic_finbert_4h_input.csv"

if FINBERT_OUTPUT_FILE.exists():
    news_finbert = pd.read_csv(FINBERT_OUTPUT_FILE)
    news_finbert["published_utc"] = pd.to_datetime(news_finbert["published_utc"], utc=True)
    print("Loaded existing FinBERT result:", FINBERT_OUTPUT_FILE)

else:
    device = 0 if torch.cuda.is_available() else -1
    print("GPU available:", torch.cuda.is_available())
    print("Device:", device)

    model_name = "ProsusAI/finbert"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    sentiment_pipe = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=None,
        truncation=True
    )

    texts = news_finbert_base["title_clean"].astype(str).tolist()

    rows = []
    batch_size = 32

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        outputs = sentiment_pipe(batch, truncation=True)

        for out in outputs:
            score_dict = {d["label"].lower(): d["score"] for d in out}

            pos = score_dict.get("positive", 0.0)
            neg = score_dict.get("negative", 0.0)
            neu = score_dict.get("neutral", 0.0)

            rows.append({
                "positive_prob": pos,
                "negative_prob": neg,
                "neutral_prob": neu,
                "finbert_score": pos - neg,
                "finbert_label": max(score_dict, key=score_dict.get)
            })

    sent_df = pd.DataFrame(rows)

    news_finbert = pd.concat(
        [news_finbert_base.reset_index(drop=True), sent_df.reset_index(drop=True)],
        axis=1
    )

    news_finbert["is_positive"] = (news_finbert["positive_prob"] > 0.6).astype(int)
    news_finbert["is_negative"] = (news_finbert["negative_prob"] > 0.6).astype(int)
    news_finbert["is_neutral"] = (news_finbert["neutral_prob"] > 0.6).astype(int)

    news_finbert.to_csv(FINBERT_OUTPUT_FILE, index=False)
    print("Saved:", FINBERT_OUTPUT_FILE)

print(news_finbert.shape)
display(news_finbert[[
    "published_utc", "title_clean", "topic",
    "finbert_label", "finbert_score",
    "positive_prob", "negative_prob", "neutral_prob"
]].head())

GPU available: True
Device: 0


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

  0%|          | 0/6561 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved: /content/drive/MyDrive/oil_news_project_outputs/news_topic_finbert_4h_input.csv
(209923, 20)


,published_utc,title_clean,topic,finbert_label,finbert_score,positive_prob,negative_prob,neutral_prob
0,2026-02-15 00:00:00+00:00,2026 Winter Olympics: Mikhail Shaidorov wins m...,other_oil_relevant,negative,-0.746291,0.054923,0.801214,0.143862
1,2026-02-15 00:00:00+00:00,Olympic gold medalist Noah Lyles runs personal...,other_oil_relevant,neutral,0.149761,0.181396,0.031635,0.786969
2,2026-02-15 00:00:00+00:00,Charley Hull uses late charge to win PIF Saudi...,opec_gulf_supply,neutral,0.274719,0.308436,0.033717,0.657847
3,2026-02-15 00:00:00+00:00,Jordan Stolz wins his 2nd Olympic speedskating...,other_oil_relevant,positive,0.654394,0.678511,0.024117,0.297372
4,2026-02-15 00:00:00+00:00,Fort Worth Chamber Summit to Address Workforce...,other_oil_relevant,neutral,0.162768,0.184276,0.021508,0.794216


In [38]:
#保存文件
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/bigquery")
OUT_DIR.mkdir(exist_ok=True)

FINBERT_SAVE_FILE = OUT_DIR / "news_topic_finbert_saved.csv"

news_finbert.to_csv(FINBERT_SAVE_FILE, index=False)

print("Saved FinBERT results to:")
print(FINBERT_SAVE_FILE)
print("Shape:", news_finbert.shape)

Saved FinBERT results to:
/content/drive/MyDrive/bigquery/news_topic_finbert_saved.csv
Shape: (209923, 20)


In [39]:
#检查 FinBERT 结果分布
print("FinBERT label distribution:")
print(news_finbert["finbert_label"].value_counts(normalize=True))

print("\nFinBERT score summary:")
display(news_finbert["finbert_score"].describe())

print("\nTopic x sentiment label:")
display(pd.crosstab(news_finbert["topic"], news_finbert["finbert_label"], normalize="index"))


FinBERT label distribution:
finbert_label
neutral     0.449150
negative    0.392711
positive    0.158139
Name: proportion, dtype: float64

FinBERT score summary:


,finbert_score
count,209923.000000
mean,-0.184454
std,0.542476
min,-0.968853
25%,-0.729297
50%,-0.079447
75%,0.109813
max,0.943752



Topic x sentiment label:


finbert_label,negative,neutral,positive
topic,,,
demand_china_growth,0.364024,0.366581,0.269395
diplomacy_ceasefire,0.392824,0.251457,0.355719
hormuz_shipping,0.393324,0.451919,0.154756
inventory_eia,0.336983,0.422697,0.240320
macro_dollar_fed,0.376672,0.397135,0.226192
military_escalation,0.467124,0.434988,0.097888
opec_gulf_supply,0.253952,0.518485,0.227564
other_oil_relevant,0.201669,0.572049,0.226281
sanctions_exports,0.352876,0.263274,0.383850


In [40]:
#按 4h 聚合 topic + FinBERT 特征
def aggregate_topic_finbert_features(df, freq="4h"):
    temp = df.copy()
    temp["published_utc"] = pd.to_datetime(temp["published_utc"], utc=True)
    temp["bucket"] = temp["published_utc"].dt.floor(freq)

    temp["relevance_score"] = pd.to_numeric(
        temp["relevance_score"], errors="coerce"
    ).fillna(1)

    temp["rel_x_sent"] = temp["relevance_score"] * temp["finbert_score"]

    # 全部新闻聚合，不分 topic
    all_agg = temp.groupby("bucket").agg(
        all_news_count=("title_clean", "size"),
        all_source_count=("source", "nunique"),
        all_sent_mean=("finbert_score", "mean"),
        all_sent_sum=("finbert_score", "sum"),
        all_sent_std=("finbert_score", "std"),
        all_pos_count=("is_positive", "sum"),
        all_neg_count=("is_negative", "sum"),
        all_neutral_count=("is_neutral", "sum"),
        all_max_pos=("positive_prob", "max"),
        all_max_neg=("negative_prob", "max"),
        all_relevance_sum=("relevance_score", "sum"),
        all_relevance_mean=("relevance_score", "mean"),
        all_rel_x_sent=("rel_x_sent", "sum")
    )

    all_agg["all_relevance_weighted_sent"] = (
        all_agg["all_rel_x_sent"] / all_agg["all_relevance_sum"].replace(0, np.nan)
    )

    all_agg = all_agg.drop(columns=["all_rel_x_sent"])

    # topic-level 聚合
    topic_agg = temp.groupby(["bucket", "topic"]).agg(
        count=("title_clean", "size"),
        source_count=("source", "nunique"),
        sent_mean=("finbert_score", "mean"),
        sent_sum=("finbert_score", "sum"),
        sent_std=("finbert_score", "std"),
        pos_count=("is_positive", "sum"),
        neg_count=("is_negative", "sum"),
        neutral_count=("is_neutral", "sum"),
        max_pos=("positive_prob", "max"),
        max_neg=("negative_prob", "max"),
        relevance_sum=("relevance_score", "sum"),
        relevance_mean=("relevance_score", "mean"),
        rel_x_sent=("rel_x_sent", "sum")
    )

    topic_agg["relevance_weighted_sent"] = (
        topic_agg["rel_x_sent"] / topic_agg["relevance_sum"].replace(0, np.nan)
    )

    topic_agg = topic_agg.drop(columns=["rel_x_sent"])

    topic_wide = topic_agg.unstack("topic")

    topic_wide.columns = [
        f"{topic}_{metric}"
        for metric, topic in topic_wide.columns
    ]

    features = all_agg.join(topic_wide, how="outer")

    features = features.fillna(0)

    # count/sum 类做 log1p
    for c in list(features.columns):
        if (
            c.endswith("_count")
            or c.endswith("_sum")
            or c in ["all_news_count", "all_relevance_sum"]
        ):
            features[f"log1p_{c}"] = np.log1p(features[c].clip(lower=0))

    features.index.name = "timestamp_utc"
    features = features.reset_index()

    return features


finbert_text_4h = aggregate_topic_finbert_features(news_finbert, freq="4h")

print("4h FinBERT text features:", finbert_text_4h.shape)
display(finbert_text_4h.head())

FINBERT_TEXT_4H_FILE = OUT_DIR / "text_features_4h_topic_finbert.csv"
finbert_text_4h.to_csv(FINBERT_TEXT_4H_FILE, index=False)
print("Saved:", FINBERT_TEXT_4H_FILE)

4h FinBERT text features: (426, 201)


,timestamp_utc,all_news_count,all_source_count,all_sent_mean,all_sent_sum,all_sent_std,all_pos_count,all_neg_count,all_neutral_count,all_max_pos,...,log1p_sanctions_exports_neutral_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 00:00:00+00:00,126,87,-0.024095,-3.035909,0.498795,18,27,61,0.935172,...,1.098612,2.484907,1.386294,1.945910,1.098612,2.639057,3.806662,2.944439,4.262680,2.995732
1,2026-02-15 04:00:00+00:00,119,84,-0.043712,-5.201700,0.509639,17,25,64,0.952422,...,1.609438,1.945910,0.000000,2.302585,2.397895,3.332205,3.871201,3.433987,3.806662,2.944439
2,2026-02-15 08:00:00+00:00,128,86,0.005525,0.707226,0.508724,20,23,66,0.911715,...,2.079442,2.772589,1.609438,2.302585,1.945910,2.708050,4.007333,4.356709,3.401197,3.295837
3,2026-02-15 12:00:00+00:00,182,127,0.069636,12.673779,0.476793,32,28,98,0.951054,...,1.609438,1.609438,2.079442,4.158883,2.397895,3.258097,3.637586,4.488636,4.158883,3.044522
4,2026-02-15 16:00:00+00:00,176,125,-0.028362,-4.991709,0.506956,21,37,88,0.948227,...,0.693147,0.000000,1.609438,4.394449,1.609438,3.135494,4.060443,3.970292,4.406719,1.945910


Saved: /content/drive/MyDrive/bigquery/text_features_4h_topic_finbert.csv


In [41]:
#准备 4h 价格和金融特征
DIRECTION_4H_FILE = OUT_DIR / "direction_dataset_4h.csv"

if "price_4h" in globals():
    price_base_4h = price_4h.copy()
    print("Using existing price_4h variable.")
else:
    price_base_4h = pd.read_csv(DIRECTION_4H_FILE)
    price_base_4h["timestamp_utc"] = pd.to_datetime(price_base_4h["timestamp_utc"], utc=True)
    print("Loaded:", DIRECTION_4H_FILE)

print(price_base_4h.shape)
print(price_base_4h.columns.tolist()[:30])
display(price_base_4h.head())

Using existing price_4h variable.
(297, 41)
['timestamp_utc', 'wti_close', 'wti_ret_0', 'wti_ret_lag1', 'wti_ret_lag2', 'wti_ret_lag3', 'wti_ret_lag6', 'wti_ret_lag12', 'wti_vol_3', 'wti_vol_6', 'wti_vol_12', 'brent_ret_0', 'brent_ret_lag1', 'brent_ret_lag2', 'brent_ret_lag3', 'gold_ret_0', 'gold_ret_lag1', 'gold_ret_lag2', 'gold_ret_lag3', 'natgas_ret_0', 'natgas_ret_lag1', 'natgas_ret_lag2', 'natgas_ret_lag3', 'spy_ret_0', 'spy_ret_lag1', 'spy_ret_lag2', 'spy_ret_lag3', 'uup_ret_0', 'uup_ret_lag1', 'uup_ret_lag2']


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,vix_ret_0,vix_ret_lag1,vix_ret_lag2,vix_ret_lag3,hour,dayofweek,hour_sin,hour_cos,future_ret_next,target_direction
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,20,6,-0.866025,0.5,0.000159,1
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0.000000,1.0,0.000000,0
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4,0,0.866025,0.5,0.004599,1
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,NaN,NaN,NaN,NaN,4,1,0.866025,0.5,0.007409,1
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,NaN,NaN,NaN,NaN,8,1,0.866025,-0.5,-0.027870,0


In [42]:
#合并 4h FinBERT 文本特征和价格特征
def identify_old_text_cols(df):
    text_prefixes = [
        "all_",
        "log1p_all_",
        "hormuz_shipping_",
        "log1p_hormuz_shipping_",
        "military_escalation_",
        "log1p_military_escalation_",
        "sanctions_exports_",
        "log1p_sanctions_exports_",
        "diplomacy_ceasefire_",
        "log1p_diplomacy_ceasefire_",
        "opec_gulf_supply_",
        "log1p_opec_gulf_supply_",
        "inventory_eia_",
        "log1p_inventory_eia_",
        "macro_dollar_fed_",
        "log1p_macro_dollar_fed_",
        "demand_china_growth_",
        "log1p_demand_china_growth_",
        "other_oil_relevant_",
        "log1p_other_oil_relevant_",
    ]

    return [
        c for c in df.columns
        if any(c.startswith(prefix) for prefix in text_prefixes)
    ]


price_base_4h = price_base_4h.copy()
price_base_4h["timestamp_utc"] = pd.to_datetime(price_base_4h["timestamp_utc"], utc=True)
finbert_text_4h["timestamp_utc"] = pd.to_datetime(finbert_text_4h["timestamp_utc"], utc=True)

old_text_cols = identify_old_text_cols(price_base_4h)

price_financial_only_4h = price_base_4h.drop(columns=old_text_cols, errors="ignore")

df_finbert_4h = price_financial_only_4h.merge(
    finbert_text_4h,
    on="timestamp_utc",
    how="left"
)

new_text_cols = [c for c in finbert_text_4h.columns if c != "timestamp_utc"]
df_finbert_4h[new_text_cols] = df_finbert_4h[new_text_cols].fillna(0)

df_finbert_4h = df_finbert_4h.replace([np.inf, -np.inf], np.nan)

# 确保 target 存在
if "target_direction" not in df_finbert_4h.columns:
    if "future_ret_next" in df_finbert_4h.columns:
        df_finbert_4h["target_direction"] = (df_finbert_4h["future_ret_next"] > 0).astype(int)
    elif "future_ret_4h" in df_finbert_4h.columns:
        df_finbert_4h["target_direction"] = (df_finbert_4h["future_ret_4h"] > 0).astype(int)
    else:
        raise ValueError("找不到 target_direction / future_ret_next / future_ret_4h")

df_finbert_4h = df_finbert_4h.dropna(subset=["target_direction"]).copy()

print("Final 4h FinBERT dataset:", df_finbert_4h.shape)
print("Target distribution:")
print(df_finbert_4h["target_direction"].value_counts(normalize=True))

display(df_finbert_4h.head())

DF_FINBERT_4H_FILE = OUT_DIR / "direction_dataset_4h_topic_finbert.csv"
df_finbert_4h.to_csv(DF_FINBERT_4H_FILE, index=False)
print("Saved:", DF_FINBERT_4H_FILE)

Final 4h FinBERT dataset: (297, 241)
Target distribution:
target_direction
1    0.572391
0    0.427609
Name: proportion, dtype: float64


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,log1p_sanctions_exports_neutral_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.609438,1.386294,4.770685,2.708050,3.218876,3.713572,3.433987,4.248495,0.000000
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.386294,2.197225,4.828314,2.197225,3.806662,3.688879,3.871201,4.007333,1.609438
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,1.098612,1.098612,2.708050,3.583519,4.574711,3.970292,3.218876,4.204693,4.369448,2.639057
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,1.098612,1.945910,2.944439,3.688879,3.091042,4.025352,3.218876,4.007333,4.465908,2.708050
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,1.098612,1.609438,3.258097,4.927254,2.833213,4.094345,4.499810,4.663439,4.304065,2.890372


Saved: /content/drive/MyDrive/bigquery/direction_dataset_4h_topic_finbert.csv


In [43]:
#定义 FinBERT text / financial feature groups
def infer_finbert_feature_groups(df, finbert_text_feature_cols):
    exclude_cols = [
        "timestamp_utc",
        "wti_close",
        "future_ret_next",
        "future_ret_4h",
        "future_direction_4h",
        "target_direction"
    ]

    numeric_cols = [
        c for c in df.columns
        if c not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    text_cols = [
        c for c in finbert_text_feature_cols
        if c in df.columns
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    financial_cols = [
        c for c in numeric_cols
        if c not in text_cols
    ]

    combined_cols = financial_cols + text_cols

    return text_cols, financial_cols, combined_cols


finbert_text_cols, financial_cols, combined_cols = infer_finbert_feature_groups(
    df_finbert_4h,
    new_text_cols
)

print("FinBERT text features:", len(finbert_text_cols))
print("Financial features:", len(financial_cols))
print("Combined features:", len(combined_cols))

print("\nExample FinBERT text cols:")
print(finbert_text_cols[:20])

print("\nExample financial cols:")
print(financial_cols[:20])

FinBERT text features: 200
Financial features: 37
Combined features: 237

Example FinBERT text cols:
['all_news_count', 'all_source_count', 'all_sent_mean', 'all_sent_sum', 'all_sent_std', 'all_pos_count', 'all_neg_count', 'all_neutral_count', 'all_max_pos', 'all_max_neg', 'all_relevance_sum', 'all_relevance_mean', 'all_relevance_weighted_sent', 'demand_china_growth_count', 'diplomacy_ceasefire_count', 'hormuz_shipping_count', 'inventory_eia_count', 'macro_dollar_fed_count', 'military_escalation_count', 'opec_gulf_supply_count']

Example financial cols:
['wti_ret_0', 'wti_ret_lag1', 'wti_ret_lag2', 'wti_ret_lag3', 'wti_ret_lag6', 'wti_ret_lag12', 'wti_vol_3', 'wti_vol_6', 'wti_vol_12', 'brent_ret_0', 'brent_ret_lag1', 'brent_ret_lag2', 'brent_ret_lag3', 'gold_ret_0', 'gold_ret_lag1', 'gold_ret_lag2', 'gold_ret_lag3', 'natgas_ret_0', 'natgas_ret_lag1', 'natgas_ret_lag2']


In [44]:
#4h 方向预测函数
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except:
    HAS_XGB = False


def get_classifier(model_name):
    if model_name == "Baseline":
        return DummyClassifier(strategy="most_frequent")

    if model_name == "Logit_L2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                class_weight="balanced"
            ))
        ])

    if model_name == "Logit_L1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                max_iter=3000,
                class_weight="balanced"
            ))
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=3,
                min_samples_leaf=5,
                random_state=42,
                class_weight="balanced"
            ))
        ])

    if model_name == "XGBoost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=200,
                max_depth=2,
                learning_rate=0.03,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            ))
        ])

    raise ValueError("Unknown model name")


def evaluate_direction_accuracy(df, feature_cols, model_name, n_splits=5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    y = data["target_direction"].astype(int).values

    if len(feature_cols) == 0:
        X = pd.DataFrame({"constant": np.zeros(len(data))})
    else:
        X = data[feature_cols].replace([np.inf, -np.inf], np.nan)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = get_classifier(model_name)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X_test)[:, 1]
        else:
            prob = pred

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        y_prob_all.extend(prob)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    out = {
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

    try:
        out["auc"] = roc_auc_score(y_true_all, y_prob_all)
    except:
        out["auc"] = np.nan

    return out

In [45]:
#跑 4h FinBERT 模型对比
feature_sets = {
    "Baseline": [],
    "FinBERT Text-only": finbert_text_cols,
    "Financial-only": financial_cols,
    "FinBERT Text + Financial": combined_cols
}

models = ["Baseline", "Logit_L2", "Logit_L1", "RandomForest"]

if HAS_XGB:
    models.append("XGBoost")

rows = []

for feature_set_name, cols in feature_sets.items():
    for model_name in models:

        if model_name == "Baseline" and feature_set_name != "Baseline":
            continue

        if model_name != "Baseline" and feature_set_name == "Baseline":
            continue

        print(
            f"Running 4h | {feature_set_name} | {model_name} | "
            f"n_features={len(cols)}"
        )

        res = evaluate_direction_accuracy(
            df_finbert_4h,
            feature_cols=cols,
            model_name=model_name,
            n_splits=5
        )

        res["horizon"] = "4h_next_period"
        res["feature_set"] = feature_set_name
        res["model"] = model_name
        res["n_features"] = len(cols)

        rows.append(res)

results_finbert_4h = pd.DataFrame(rows)

results_finbert_4h = results_finbert_4h[
    [
        "horizon",
        "feature_set",
        "model",
        "n_features",
        "accuracy",
        "balanced_accuracy",
        "f1",
        "auc",
        "up_rate_true",
        "up_rate_pred",
        "n_eval",
        "confusion_matrix"
    ]
].sort_values(["accuracy", "balanced_accuracy"], ascending=False)

display(results_finbert_4h)

RESULTS_FINBERT_4H_FILE = OUT_DIR / "direction_results_4h_topic_finbert.csv"
results_finbert_4h.to_csv(RESULTS_FINBERT_4H_FILE, index=False)
print("Saved:", RESULTS_FINBERT_4H_FILE)

Running 4h | Baseline | Baseline | n_features=0
Running 4h | FinBERT Text-only | Logit_L2 | n_features=200
Running 4h | FinBERT Text-only | Logit_L1 | n_features=200
Running 4h | FinBERT Text-only | RandomForest | n_features=200
Running 4h | FinBERT Text-only | XGBoost | n_features=200
Running 4h | Financial-only | Logit_L2 | n_features=37
Running 4h | Financial-only | Logit_L1 | n_features=37
Running 4h | Financial-only | RandomForest | n_features=37
Running 4h | Financial-only | XGBoost | n_features=37
Running 4h | FinBERT Text + Financial | Logit_L2 | n_features=237
Running 4h | FinBERT Text + Financial | Logit_L1 | n_features=237
Running 4h | FinBERT Text + Financial | RandomForest | n_features=237
Running 4h | FinBERT Text + Financial | XGBoost | n_features=237


,horizon,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,n_eval,confusion_matrix
0,4h_next_period,Baseline,Baseline,0,0.559184,0.506738,0.682353,0.506738,0.587755,0.800000,245,"[[21, 80], [28, 116]]"
3,4h_next_period,FinBERT Text-only,RandomForest,200,0.551020,0.493881,0.682081,0.463009,0.587755,0.824490,245,"[[17, 84], [26, 118]]"
11,4h_next_period,FinBERT Text + Financial,RandomForest,237,0.546939,0.487452,0.681948,0.454414,0.587755,0.836735,245,"[[15, 86], [25, 119]]"
4,4h_next_period,FinBERT Text-only,XGBoost,200,0.518367,0.479407,0.631250,0.456271,0.587755,0.718367,245,"[[26, 75], [43, 101]]"
9,4h_next_period,FinBERT Text + Financial,Logit_L2,237,0.510204,0.485767,0.600000,0.468716,0.587755,0.636735,245,"[[35, 66], [54, 90]]"
10,4h_next_period,FinBERT Text + Financial,Logit_L1,237,0.506122,0.488208,0.584192,0.456202,0.587755,0.600000,245,"[[39, 62], [59, 85]]"
2,4h_next_period,FinBERT Text-only,Logit_L1,200,0.493878,0.482226,0.560284,0.452214,0.587755,0.563265,245,"[[42, 59], [65, 79]]"
12,4h_next_period,FinBERT Text + Financial,XGBoost,237,0.489796,0.446232,0.615385,0.405047,0.587755,0.738776,245,"[[20, 81], [44, 100]]"
5,4h_next_period,Financial-only,Logit_L2,37,0.485714,0.460499,0.580000,0.458024,0.587755,0.636735,245,"[[32, 69], [57, 87]]"
1,4h_next_period,FinBERT Text-only,Logit_L2,200,0.473469,0.464865,0.534296,0.470503,0.587755,0.542857,245,"[[42, 59], [70, 74]]"


Saved: /content/drive/MyDrive/bigquery/direction_results_4h_topic_finbert.csv


In [46]:
#和你之前的 topic-count 结果对比
OLD_RESULTS_FILE = OUT_DIR / "direction_prediction_results_1h_4h.csv"

if OLD_RESULTS_FILE.exists():
    old_results = pd.read_csv(OLD_RESULTS_FILE)

    old_4h = old_results[old_results["horizon"] == "4h_next_period"].copy()

    compare_old_new = pd.concat([
        old_4h.assign(version="Topic Count only"),
        results_finbert_4h.assign(version="Topic + FinBERT")
    ], ignore_index=True)

    display(
        compare_old_new[
            [
                "version",
                "horizon",
                "feature_set",
                "model",
                "n_features",
                "accuracy",
                "balanced_accuracy",
                "f1",
                "auc",
                "up_rate_true",
                "up_rate_pred"
            ]
        ].sort_values(["version", "accuracy"], ascending=[True, False])
    )

    compare_old_new.to_csv(
        OUT_DIR / "compare_topic_count_vs_finbert_4h.csv",
        index=False
    )
else:
    print("No old result file found.")

No old result file found.


In [47]:
#看 FinBERT text-only 里哪些特征重要
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def fit_l1_coefficients(df, feature_cols, C=0.5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            max_iter=3000,
            class_weight="balanced"
        ))
    ])

    model.fit(X, y)

    coef = model.named_steps["clf"].coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "coef": coef,
        "abs_coef": np.abs(coef)
    }).sort_values("abs_coef", ascending=False)

    return model, coef_df


l1_text_model, finbert_coef_text = fit_l1_coefficients(
    df_finbert_4h,
    finbert_text_cols,
    C=0.5
)

display(finbert_coef_text.head(40))

finbert_coef_text.to_csv(
    OUT_DIR / "finbert_text_l1_coefficients_4h.csv",
    index=False
)

,feature,coef,abs_coef
149,log1p_inventory_eia_source_count,0.964077,0.964077
29,other_oil_relevant_source_count,-0.648244,0.648244
162,log1p_other_oil_relevant_sent_sum,-0.608905,0.608905
113,diplomacy_ceasefire_relevance_mean,-0.584437,0.584437
110,other_oil_relevant_relevance_sum,0.539627,0.539627
70,inventory_eia_neg_count,-0.452639,0.452639
119,other_oil_relevant_relevance_mean,-0.413620,0.413620
181,log1p_sanctions_exports_neg_count,-0.345876,0.345876
128,other_oil_relevant_relevance_weighted_sent,0.318797,0.318797
72,military_escalation_neg_count,-0.279783,0.279783


In [48]:
#你应该怎么解读结果
display(
    results_finbert_4h[
        [
            "feature_set",
            "model",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ]

)


,feature_set,model,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred
0,Baseline,Baseline,0.559184,0.506738,0.506738,0.587755,0.800000
3,FinBERT Text-only,RandomForest,0.551020,0.493881,0.463009,0.587755,0.824490
11,FinBERT Text + Financial,RandomForest,0.546939,0.487452,0.454414,0.587755,0.836735
4,FinBERT Text-only,XGBoost,0.518367,0.479407,0.456271,0.587755,0.718367
9,FinBERT Text + Financial,Logit_L2,0.510204,0.485767,0.468716,0.587755,0.636735
10,FinBERT Text + Financial,Logit_L1,0.506122,0.488208,0.456202,0.587755,0.600000
2,FinBERT Text-only,Logit_L1,0.493878,0.482226,0.452214,0.587755,0.563265
12,FinBERT Text + Financial,XGBoost,0.489796,0.446232,0.405047,0.587755,0.738776
5,Financial-only,Logit_L2,0.485714,0.460499,0.458024,0.587755,0.636735
1,FinBERT Text-only,Logit_L2,0.473469,0.464865,0.470503,0.587755,0.542857


In [49]:
# 删除 other_oil_relevant 相关特征
finbert_text_cols_no_other = [
    c for c in finbert_text_cols
    if not c.startswith("other_oil_relevant_")
    and not c.startswith("log1p_other_oil_relevant_")
]

combined_cols_no_other = financial_cols + finbert_text_cols_no_other

print("Original FinBERT text features:", len(finbert_text_cols))
print("No-other FinBERT text features:", len(finbert_text_cols_no_other))
print("Combined no-other features:", len(combined_cols_no_other))

Original FinBERT text features: 200
No-other FinBERT text features: 180
Combined no-other features: 217


In [50]:
feature_sets_no_other = {
    "FinBERT Text-only no other": finbert_text_cols_no_other,
    "Financial-only": financial_cols,
    "FinBERT Text + Financial no other": combined_cols_no_other
}

models_to_run = ["Logit_L2", "Logit_L1", "RandomForest"]

if HAS_XGB:
    models_to_run.append("XGBoost")

rows_no_other = []

for feature_set_name, cols in feature_sets_no_other.items():
    for model_name in models_to_run:

        print(f"Running | {feature_set_name} | {model_name} | n_features={len(cols)}")

        res = evaluate_direction_accuracy(
            df_finbert_4h,
            feature_cols=cols,
            model_name=model_name,
            n_splits=5
        )

        res["feature_set"] = feature_set_name
        res["model"] = model_name
        res["n_features"] = len(cols)

        rows_no_other.append(res)

results_no_other = pd.DataFrame(rows_no_other)

display(
    results_no_other[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].sort_values(["balanced_accuracy", "auc"], ascending=False)
)

results_no_other.to_csv(
    OUT_DIR / "direction_results_4h_finbert_no_other.csv",
    index=False
)

Running | FinBERT Text-only no other | Logit_L2 | n_features=180
Running | FinBERT Text-only no other | Logit_L1 | n_features=180
Running | FinBERT Text-only no other | RandomForest | n_features=180
Running | FinBERT Text-only no other | XGBoost | n_features=180
Running | Financial-only | Logit_L2 | n_features=37
Running | Financial-only | Logit_L1 | n_features=37
Running | Financial-only | RandomForest | n_features=37
Running | Financial-only | XGBoost | n_features=37
Running | FinBERT Text + Financial no other | Logit_L2 | n_features=217
Running | FinBERT Text + Financial no other | Logit_L1 | n_features=217
Running | FinBERT Text + Financial no other | RandomForest | n_features=217
Running | FinBERT Text + Financial no other | XGBoost | n_features=217


,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred
0,FinBERT Text-only no other,Logit_L2,180,0.493878,0.482226,0.560284,0.468853,0.587755,0.563265
2,FinBERT Text-only no other,RandomForest,180,0.542857,0.481023,0.681818,0.461840,0.587755,0.848980
1,FinBERT Text-only no other,Logit_L1,180,0.493878,0.480748,0.563380,0.455583,0.587755,0.571429
10,FinBERT Text + Financial no other,RandomForest,217,0.534694,0.474079,0.674286,0.437019,0.587755,0.840816
3,FinBERT Text-only no other,XGBoost,180,0.497959,0.460568,0.611987,0.412129,0.587755,0.706122
4,Financial-only,Logit_L2,37,0.485714,0.460499,0.580000,0.458024,0.587755,0.636735
9,FinBERT Text + Financial no other,Logit_L1,217,0.465306,0.447573,0.546713,0.445545,0.587755,0.591837
8,FinBERT Text + Financial no other,Logit_L2,217,0.477551,0.446163,0.584416,0.452833,0.587755,0.669388
5,Financial-only,Logit_L1,37,0.469388,0.445132,0.563758,0.454552,0.587755,0.628571
7,Financial-only,XGBoost,37,0.473469,0.429387,0.603077,0.389508,0.587755,0.738776


In [51]:
core_topics = [
    "inventory_eia",
    "diplomacy_ceasefire",
    "military_escalation",
    "hormuz_shipping",
    "sanctions_exports"
]

def is_core_topic_feature(col, topics):
    for topic in topics:
        if col.startswith(topic + "_") or col.startswith("log1p_" + topic + "_"):
            return True
    return False

finbert_text_cols_core = [
    c for c in finbert_text_cols
    if is_core_topic_feature(c, core_topics)
]

combined_cols_core = financial_cols + finbert_text_cols_core

print("Core FinBERT text features:", len(finbert_text_cols_core))
print(finbert_text_cols_core[:30])

Core FinBERT text features: 100
['diplomacy_ceasefire_count', 'hormuz_shipping_count', 'inventory_eia_count', 'military_escalation_count', 'sanctions_exports_count', 'diplomacy_ceasefire_source_count', 'hormuz_shipping_source_count', 'inventory_eia_source_count', 'military_escalation_source_count', 'sanctions_exports_source_count', 'diplomacy_ceasefire_sent_mean', 'hormuz_shipping_sent_mean', 'inventory_eia_sent_mean', 'military_escalation_sent_mean', 'sanctions_exports_sent_mean', 'diplomacy_ceasefire_sent_sum', 'hormuz_shipping_sent_sum', 'inventory_eia_sent_sum', 'military_escalation_sent_sum', 'sanctions_exports_sent_sum', 'diplomacy_ceasefire_sent_std', 'hormuz_shipping_sent_std', 'inventory_eia_sent_std', 'military_escalation_sent_std', 'sanctions_exports_sent_std', 'diplomacy_ceasefire_pos_count', 'hormuz_shipping_pos_count', 'inventory_eia_pos_count', 'military_escalation_pos_count', 'sanctions_exports_pos_count']


In [52]:
feature_sets_core = {
    "Core FinBERT Text-only": finbert_text_cols_core,
    "Financial-only": financial_cols,
    "Core FinBERT Text + Financial": combined_cols_core
}

rows_core = []

for feature_set_name, cols in feature_sets_core.items():
    for model_name in models_to_run:

        print(f"Running | {feature_set_name} | {model_name} | n_features={len(cols)}")

        res = evaluate_direction_accuracy(
            df_finbert_4h,
            feature_cols=cols,
            model_name=model_name,
            n_splits=5
        )

        res["feature_set"] = feature_set_name
        res["model"] = model_name
        res["n_features"] = len(cols)

        rows_core.append(res)

results_core = pd.DataFrame(rows_core)

display(
    results_core[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].sort_values(["balanced_accuracy", "auc"], ascending=False)
)

results_core.to_csv(
    OUT_DIR / "direction_results_4h_finbert_core_topics.csv",
    index=False
)

Running | Core FinBERT Text-only | Logit_L2 | n_features=100
Running | Core FinBERT Text-only | Logit_L1 | n_features=100
Running | Core FinBERT Text-only | RandomForest | n_features=100
Running | Core FinBERT Text-only | XGBoost | n_features=100
Running | Financial-only | Logit_L2 | n_features=37
Running | Financial-only | Logit_L1 | n_features=37
Running | Financial-only | RandomForest | n_features=37
Running | Financial-only | XGBoost | n_features=37
Running | Core FinBERT Text + Financial | Logit_L2 | n_features=137
Running | Core FinBERT Text + Financial | Logit_L1 | n_features=137
Running | Core FinBERT Text + Financial | RandomForest | n_features=137
Running | Core FinBERT Text + Financial | XGBoost | n_features=137


,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred
8,Core FinBERT Text + Financial,Logit_L2,137,0.530612,0.506085,0.617940,0.473804,0.587755,0.640816
2,Core FinBERT Text-only,RandomForest,100,0.555102,0.500309,0.682216,0.446920,0.587755,0.812245
9,Core FinBERT Text + Financial,Logit_L1,137,0.530612,0.500172,0.627832,0.500138,0.587755,0.673469
0,Core FinBERT Text-only,Logit_L2,100,0.518367,0.498625,0.598639,0.474422,0.587755,0.612245
1,Core FinBERT Text-only,Logit_L1,100,0.502041,0.489171,0.570423,0.473460,0.587755,0.571429
10,Core FinBERT Text + Financial,RandomForest,137,0.542857,0.482501,0.680000,0.436469,0.587755,0.840816
4,Financial-only,Logit_L2,37,0.485714,0.460499,0.580000,0.458024,0.587755,0.636735
5,Financial-only,Logit_L1,37,0.469388,0.445132,0.563758,0.454552,0.587755,0.628571
3,Core FinBERT Text-only,XGBoost,100,0.477551,0.437294,0.600000,0.416460,0.587755,0.718367
7,Financial-only,XGBoost,37,0.473469,0.429387,0.603077,0.389508,0.587755,0.738776


In [53]:
all_result_versions = []

if "results_finbert_4h" in globals():
    temp = results_finbert_4h.copy()
    temp["version"] = "All FinBERT topics"
    all_result_versions.append(temp)

if "results_no_other" in globals():
    temp = results_no_other.copy()
    temp["version"] = "Drop other_oil_relevant"
    all_result_versions.append(temp)

if "results_core" in globals():
    temp = results_core.copy()
    temp["version"] = "Core topics only"
    all_result_versions.append(temp)

compare_topic_versions = pd.concat(all_result_versions, ignore_index=True)

display(
    compare_topic_versions[
        [
            "version",
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].sort_values(["balanced_accuracy", "auc"], ascending=False)
)

compare_topic_versions.to_csv(
    OUT_DIR / "compare_finbert_topic_versions_4h.csv",
    index=False
)

,version,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred
0,All FinBERT topics,Baseline,Baseline,0,0.559184,0.506738,0.682353,0.506738,0.587755,0.800000
33,Core topics only,Core FinBERT Text + Financial,Logit_L2,137,0.530612,0.506085,0.617940,0.473804,0.587755,0.640816
27,Core topics only,Core FinBERT Text-only,RandomForest,100,0.555102,0.500309,0.682216,0.446920,0.587755,0.812245
34,Core topics only,Core FinBERT Text + Financial,Logit_L1,137,0.530612,0.500172,0.627832,0.500138,0.587755,0.673469
25,Core topics only,Core FinBERT Text-only,Logit_L2,100,0.518367,0.498625,0.598639,0.474422,0.587755,0.612245
1,All FinBERT topics,FinBERT Text-only,RandomForest,200,0.551020,0.493881,0.682081,0.463009,0.587755,0.824490
26,Core topics only,Core FinBERT Text-only,Logit_L1,100,0.502041,0.489171,0.570423,0.473460,0.587755,0.571429
5,All FinBERT topics,FinBERT Text + Financial,Logit_L1,237,0.506122,0.488208,0.584192,0.456202,0.587755,0.600000
2,All FinBERT topics,FinBERT Text + Financial,RandomForest,237,0.546939,0.487452,0.681948,0.454414,0.587755,0.836735
4,All FinBERT topics,FinBERT Text + Financial,Logit_L2,237,0.510204,0.485767,0.600000,0.468716,0.587755,0.636735
